# Финальная модель — Левин
Итог работы из [model_levin.ipynb](model_levin.ipynb) и [clients_levin.ipynb](clients_levin.ipynb). Здесь — **финальная конфигурация** и её честная оценка на новых периодах.

**Конфигурация**
- **Модель:** один LightGBM (мелкие деревья: `num_leaves=4`, `n_estimators=400`, `learning_rate=0.02`, `min_child_samples=100`), без ансамблей, без взвешивания классов и сэмплинга (ранжирование не улучшают, калибровку портят).
- **Признаки:** без `duration` (утечка), без прокси периода (`month`, `day`, `contact`, `contact_unknown`), без года, **без `campaign`** (формально «из будущего», вклад ≈ 0,005 AUC); плюс **история клиента на момент звонка**: число прошлых покупок, флаг «покупал», дней с последней покупки, глубина цепочки.
- **Валидация:** только по времени (forward-chaining). Группировка по клиентам не нужна — в боевом скоринге вся прошлая база известна.

## 0. Данные и признаки

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
import utils_levin as ul
pd.set_option('display.width', 140, 'display.max_columns', 40)

raw, y, year, period = ul.load(); n = len(y)
X_old = ul.build(raw, drop=ul.NP_DROP)                 # прежний набор (с campaign, без истории)
X = ul.final_features(raw, year, y)                    # финальный набор
print('строк:', n, '| доля yes:', y.mean().round(3), '| признаков:', X.shape[1])
print('новые признаки:', [c for c in X.columns if c.startswith('h_')], '| убран: campaign')

строк: 45211 | доля yes: 0.117 | признаков: 23
новые признаки: ['h_depth', 'h_n_buys', 'h_ever_bought', 'h_days_since_buy'] | убран: campaign


**Проверка истории:** конверсия по числу прошлых покупок. Признак информативный, но таких клиентов мало — поэтому общий прирост качества будет небольшим.

In [2]:
g = pd.Series(y).groupby(X.h_n_buys.clip(upper=3).values).agg(['mean', 'size'])
g.index = ['0', '1', '2', '3+']; display(g.round(3).rename(columns={'mean': 'конверсия', 'size': 'строк'}).T)
print('строк с историей (цепочка из 1+ прошлых записей):', int((X.h_depth > 0).sum()))

,0,1,2,3+
конверсия,0.097,0.581,0.769,0.792
строк,43507.000,1322.000,286.000,96.000


строк с историей (цепочка из 1+ прошлых записей): 7023


## 1. Валидация по времени
Forward-chaining: учимся на прошлом, тестируем на следующем куске. Сравниваем прежний набор и финальный. Среднее по фолдам — главное число; pooled AUC (все тестовые фолды вместе) завышен, т.к. смешивает периоды с разной базовой конверсией.

In [3]:
rows = []
for k in (5, 10):
    sp = ul.time_splits(n, k); te = np.concatenate([t for _, t in sp])
    for name, F in [('прежний (с campaign)', X_old), ('финальный', X)]:
        m, oof, _ = ul.run_cv(ul.make_lgb, F, y, sp)
        rows.append({'k': k, 'набор': name, 'AUC (по фолдам)': m.auc.mean(), 'PR-AUC': m.pr.mean(), 'Lift@10%': m.lift10.mean(),
                     'AUC pooled': roc_auc_score(y[te], oof[te])})
display(pd.DataFrame(rows).round(3))

,k,набор,AUC (по фолдам),PR-AUC,Lift@10%,AUC pooled
0,5,прежний (с campaign),0.602,0.194,1.514,0.749
1,5,финальный,0.605,0.195,1.631,0.749
2,10,прежний (с campaign),0.601,0.178,1.655,0.760
3,10,финальный,0.601,0.182,1.682,0.759


Разница между наборами в пределах шума (≈ 0,003 AUC), но финальный набор не хуже, честнее (нет «признака из будущего») и даёт больше на клиентах с историей (см. ниже).

## 2. Тест на актуальном периоде (2009+)
Ранний 2008 год — период массовых рассылок (конверсия 3–6%, 61% строк), для текущей ситуации он не репрезентативен. Тестируем только на 2009+ (6 блоков по времени, берём блоки 2–5), а обучаем на всём, что раньше. Проверяем: нужно ли выбрасывать/занижать 2008 в обучении.

In [4]:
c09 = int(np.argmax(period >= 200901)); c0811 = int(np.argmax(period >= 200811))
blocks = np.array_split(np.arange(c09, n), 6)
print('2008 год:', c09, 'строк | 2009+:', n - c09, 'строк')

def run(start=0, w=None):
    rows = []
    for bi in range(2, 6):
        te = blocks[bi]; tr = np.arange(start, te[0])
        p = ul.make_lgb(X).fit(X.iloc[tr], y[tr], sample_weight=None if w is None else w[tr]).predict_proba(X.iloc[te])[:, 1]
        rows.append((roc_auc_score(y[te], p), average_precision_score(y[te], p), ul.lift_at(y[te], p), len(tr)))
    return pd.DataFrame(rows, columns=['auc', 'pr', 'lift', 'ntrain'])

variants = {'все данные с 2008-05': run(0), 'без мая–авг 2008': run(c0811), 'только 2009+': run(c09),
            'все, вес по давности 0,2→1': run(0, np.linspace(0.2, 1, n)), 'все, вес 2008 = 0,3': run(0, np.where(np.arange(n) < c09, 0.3, 1.0))}
res = pd.DataFrame({k: {'AUC': v.auc.mean(), 'AUC min блока': v.auc.min(), 'AUC max блока': v.auc.max(), 'PR-AUC': v.pr.mean(),
                        'Lift@10%': v.lift.mean(), 'train, строк': v.ntrain.mean()} for k, v in variants.items()}).T
display(res.round(3))

2008 год: 27729 строк | 2009+: 17482 строк


,AUC,AUC min блока,AUC max блока,PR-AUC,Lift@10%,"train, строк"
все данные с 2008-05,0.630,0.572,0.717,0.387,1.883,37927.75
без мая–авг 2008,0.634,0.566,0.726,0.388,1.875,13809.75
только 2009+,0.628,0.565,0.703,0.387,1.824,10198.75
"все, вес по давности 0,2→1",0.632,0.570,0.719,0.389,1.852,37927.75
"все, вес 2008 = 0,3",0.635,0.572,0.728,0.393,1.919,37927.75


Все варианты в пределах ≈ 0,007 AUC при разбросе между блоками ±0,06 — отличить их нельзя. 2008 год не вредит (модель учится на нём общим зависимостям), поэтому **обучаем на всех данных**; вес 2008 = 0,3 — возможный, но не обязательный бонус (+0,006, в пределах шума).

## 3. Качество по сегментам и важность признаков

In [5]:
sp = ul.time_splits(n, 10); te = np.concatenate([t for _, t in sp[4:]])      # поздние тестовые фолды
_, oof, _ = ul.run_cv(ul.make_lgb, X, y, sp[4:])
hist = (X.h_depth > 0).values
seg = pd.DataFrame({'сегмент': ['все', 'есть история', 'нет истории'],
    'строк в тесте': [len(te), hist[te].sum(), (~hist[te]).sum()],
    'доля yes': [y[te].mean(), y[te][hist[te]].mean(), y[te][~hist[te]].mean()],
    'AUC': [roc_auc_score(y[te], oof[te]), roc_auc_score(y[te][hist[te]], oof[te][hist[te]]), roc_auc_score(y[te][~hist[te]], oof[te][~hist[te]])]})
display(seg.round(3))

m = ul.make_lgb(X, importance_type='gain').fit(X, y)
imp = pd.Series(m.feature_importances_, X.columns); imp = (imp / imp.sum()).sort_values(ascending=False)
display(imp.head(10).round(3).to_frame('доля gain').T)

,сегмент,строк в тесте,доля yes,AUC
0,все,24660,0.175,0.756
1,есть история,7023,0.237,0.829
2,нет истории,17637,0.151,0.702


,h_days_since_buy,h_n_buys,n_loans,age,pdays_clean,poutcome,age_bin,h_ever_bought,balance,prev_success
доля gain,0.238,0.152,0.115,0.095,0.094,0.056,0.034,0.034,0.034,0.033


## 4. Вердикт

**Финальная конфигурация:** LightGBM (мелкие деревья) на признаках без `duration`, без прокси периода, без `campaign`, плюс история клиента на момент звонка; обучение на всех данных; валидация по времени.

**Качество на новых периодах (честная оценка):**
- time split по всему датасету: **AUC ≈ 0,60**, PR-AUC ≈ 0,20, Lift@10% ≈ 1,6;
- тест на актуальном периоде (2009+): **AUC ≈ 0,63**, PR-AUC ≈ 0,39, Lift@10% ≈ 1,8–1,9;
- прежние 0,744 (случайная CV) и 0,725 (block) завышены: модель частично угадывала период. Разброс по блокам времени большой (0,57–0,72).

**По сегментам:**
- клиенты с историей (≈ 15% строк): сильный сегмент, AUC ≈ 0,83 против ≈ 0,70 без истории (pooled по поздним фолдам, поэтому абсолютные значения завышены, важна разница); конверсия у прошлых покупателей 58–79%, в основном это заслуга `poutcome` и числа прошлых покупок;
- клиенты без истории (холодный обзвон): слабое место, AUC ≈ 0,70 pooled и ниже на новых месяцах;
- среди уже покупавших отличить будущих повторных покупателей почти нечем (AUC ≈ 0,66).

**Что дало и не дало:**
- история клиента: небольшой прирост (0…+0,003 AUC в целом, до +0,01 на клиентах с историей) — цепочки короткие, а `poutcome` уже несёт главное;
- убрать `campaign`: качество не страдает, исчезает признак «из будущего»;
- выбросить/занизить 2008: без заметного эффекта;
- реальные значения стоимостей звонка/подписки (`r`, `c`) нужны от заказчика для выбора порога (см. раздел 9 в `model_levin.ipynb`).